코드 basic-classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 학습-시험 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 모델 학습
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 예측 및 평가
y_pred = model.predict(X_test)
print(f"정확도: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

# 계수 해석
import numpy as np
for name, coef in zip(feature_names, model.coef_[0]):
    print(f"{name}: β={coef:.3f}, 오즈비={np.exp(coef):.3f}")

코드 regression-diagnostics

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt

# 모델 적합
X_with_const = sm.add_constant(X)
model = sm.OLS(y, X_with_const).fit()
print(model.summary())

# 잔차 분석
residuals = model.resid
fitted = model.fittedvalues

# 1) 잔차 vs 적합값 (등분산성 확인)
plt.scatter(fitted, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted'); plt.ylabel('Residuals')
plt.show()

# 2) Q-Q plot (정규성 확인)
sm.qqplot(residuals, line='s'); plt.show()

코드 regularization-comparison

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error
import numpy as np

# 정규화 강도 후보
lambdas = np.logspace(-3, 2, 50)

results = {}
for name, ModelClass in [('Ridge', Ridge), ('Lasso', Lasso),
                          ('ElasticNet', ElasticNet)]:
    val_mse = []
    for alpha in lambdas:
        model = ModelClass(alpha=alpha, max_iter=5000)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        val_mse.append(mean_squared_error(y_val, pred))
    best_idx = np.argmin(val_mse)
    results[name] = (lambdas[best_idx], val_mse[best_idx])
    print(f"{name}: 최적 λ={lambdas[best_idx]:.4f}, 검증 MSE={val_mse[best_idx]:.4f}")

코드 preprocessing

In [ ]:
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer

mecab = Mecab()

def korean_tokenizer(text):
    pos_tags = mecab.pos(text)
    # 명사·동사·형용사만 추출
    tokens = [word for word, tag in pos_tags
              if tag.startswith(('NN', 'VV', 'VA'))]
    return tokens

vectorizer = TfidfVectorizer(
    tokenizer=korean_tokenizer,
    max_features=5000,      # 상위 5,000개 어휘
    min_df=5,                # 5개 이상 문서에 출현
    max_df=0.7,              # 70% 이상 문서에 출현하는 단어 제외
    ngram_range=(1, 2)       # 단어와 2-그램 모두 포함
)

X = vectorizer.fit_transform(articles_text)
print(f"문서 수: {X.shape[0]}, 어휘 수: {X.shape[1]}")

코드 frame-classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# 학습-시험 분할 (계층적 샘플링으로 클래스 균형 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# L1 정규화 다항 로지스틱 회귀
model = LogisticRegression(
    penalty='l1',
    solver='saga',
    multi_class='multinomial',
    C=1.0,                  # C = 1/λ; C가 작을수록 강한 정규화
    max_iter=2000,
    random_state=42
)
model.fit(X_train, y_train)

# 평가
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred,
      target_names=['갈등', '책임', '도덕성', '위기']))
print("\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred))

코드 interpret-frames

In [ ]:
import pandas as pd

feature_names = vectorizer.get_feature_names_out()
frame_names = ['갈등', '책임', '도덕성', '위기']

for i, frame in enumerate(frame_names):
    coefs = model.coef_[i]
    top_idx = np.argsort(coefs)[-15:][::-1]   # 상위 15개
    top_words = [(feature_names[j], coefs[j]) for j in top_idx]
    print(f"\n[{frame} 프레임 상위 신호 단어]")
    for word, coef in top_words:
        print(f"  {word}: {coef:.3f}")